In [39]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, GPT2LMHeadModel

## 1 Tokenization (Pre-processing)

Raw string (`text`) → integer token IDs. Not a neural layer — it's a lookup table built from training data vocabulary. Output is a 1D integer tensor.

### Math

$$\mathbf{x} \in \mathbb{Z}^T$$

> $T$ = sequence length (each token is an integer index into the vocabulary $V$)

In [40]:
# Define the text to be tokenized
text = "Hello there, who are you?"

# Load the GPT-2 tokenizer
Tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Tokenize the text and return PyTorch tensors
tokens = Tokenizer(text, return_tensor="pt")


input_ids = torch.tensor(tokens['input_ids'])

vocab_size = Tokenizer.vocab_size
d_model = 768  # GPT-2 small embedding dimension
T = len(tokens['input_ids'])

# In PyTorch:
print(f"Tokens (Type:{type(tokens['input_ids'])}):")
print(f"Token Input ID's: {tokens['input_ids']}")
print(f"input_ids: {input_ids}")

Tokens (Type:<class 'list'>):
Token Input ID's: [15496, 612, 11, 508, 389, 345, 30]
input_ids: tensor([15496,   612,    11,   508,   389,   345,    30])


## 2 Token Embedding (Embedding Layer)

Integer IDs → dense float vectors. The embedding matrix $W_E$ is a learnable lookup table. Row $i$ is the vector representation of token $i$. This is where token meaning lives initially.

### Math

$$W_E \in \mathbb{R}^{|V| \times d_{\text{model}}}$$

$$\mathbf{x}_{\text{embed}} = W_E[\text{token\_ids}]$$

> Output shape: $(B, T, d_{\text{model}})$ — $B$=batch, $T$=seq, $d$=embed dim

In [41]:
# Embedding Layer
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model): #(50257, 768) or (|V| X d_model)
        super(TokenEmbedding, self).__init__()
        
        self.W_E = nn.Embedding(num_embeddings=vocab_size,
                                embedding_dim=d_model
                            )

    def forward(self, input_ids):
        return self.W_E(input_ids)

x_tok = TokenEmbedding(vocab_size=vocab_size, d_model=d_model)
type(x_tok.W_E)

torch.nn.modules.sparse.Embedding

## 3 Positional Encoding

Attention has no inherent notion of order — position encodings inject "where in the sequence am I?" into each token vector. Added element-wise to token embeddings. GPT-2 uses learned; original Transformer used sinusoidal.

### Math

$$W_P \in \mathbb{R}^{T_{\max} \times d_{\text{model}}}$$

$$\mathbf{x} = W_E[\text{ids}] + W_P[\text{positions}]$$

> Output still: $(B, T, d_{\text{model}})$

In [42]:
T_max = 1024  # GPT-2 maximum sequence length

# Positional Encoding Layer
class PositionalEncoding(nn.Module):
    def __init__(self, T_max, embedding_dim):  # (T_max, d_model) = (1024, 768)
        super(PositionalEncoding, self).__init__()
        self.W_P = nn.Embedding(
            num_embeddings=T_max,       # one vector per POSITION, not per vocab token
            embedding_dim=embedding_dim
        )

    def forward(self, positions):
        return self.W_P(positions)      # (T, d_model)

positional_encoding = PositionalEncoding(T_max=T_max, embedding_dim=d_model)

print(positional_encoding.W_P.weight)

Parameter containing:
tensor([[ 0.8903, -0.5177,  0.3356,  ...,  0.1902,  0.7698, -1.2632],
        [-0.2102,  0.9888,  0.6619,  ...,  0.8001,  0.1784, -1.9218],
        [ 1.2659, -0.8249,  0.6627,  ..., -0.7125,  0.1908, -1.6263],
        ...,
        [-1.0091,  0.6665, -0.1082,  ...,  0.6474, -0.5715, -1.8075],
        [ 1.7170, -1.6798, -0.4519,  ..., -1.7305, -0.4365,  1.0324],
        [-0.8283, -0.8512, -0.2322,  ..., -0.8984, -0.4544,  0.3637]],
       requires_grad=True)


## 4A Layer Norm (Normalization)

Stabilizes training by normalizing each token's activation vector to zero mean and unit variance, then applying a learned scale ($\gamma$) and shift ($\beta$). Applied before attention in GPT-2 (pre-norm style).

### Math

$$\text{LayerNorm}(\mathbf{x}) = \gamma \cdot \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where:

$$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d}(x_i - \mu)^2$$

> Output still: $(B, T, d_{\text{model}})$

In [43]:
class Norm4a(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super(Norm4a, self).__init__()
        self.ln = nn.LayerNorm(
            normalized_shape=d_model,
            eps=eps
        )
    def forward(self, x):
        return self.ln(x)

positional_encoding_norm = Norm4a(d_model=d_model)
normalized_output = positional_encoding_norm(positional_encoding.W_P.weight)

## 4B Multi-Head Self-Attention (Attention + Linear)

The core of transformers. Each token attends to all other tokens (in decoder: only previous tokens via causal mask). "Multi-head" means we run $H$ parallel attention operations in subspaces of $d_{\text{model}}$, then concatenate. Uses 4 Linear layers internally: $W_Q$, $W_K$, $W_V$ (projections in), $W_O$ (projection out).

### Math

$$Q = xW_Q, \quad K = xW_K, \quad V = xW_V$$

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

$$\text{MultiHead} = \text{concat}(\text{head}_1, \ldots, \text{head}_H) \, W_O$$

> $W_Q, W_K, W_V \in \mathbb{R}^{d \times d}$, $W_O \in \mathbb{R}^{d \times d}$

In [44]:
num_heads = 12

In [45]:
# Compute proper input: run our 7 tokens through the full pipeline so far
embedded_tokens = x_tok(input_ids)                                        # (T, d_model) = (7, 768)
pos_encoded = embedded_tokens + positional_encoding(torch.arange(T))      # (T, d_model)
normalized_output = positional_encoding_norm(pos_encoded)                  # (T, d_model)

class MultiheadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiheadAttention, self).__init__()
        self.MHA = nn.MultiheadAttention(
            embed_dim=d_model, 
            num_heads=num_heads,
            dropout=0.0,          # 0.0 for inference with pretrained weights
            batch_first=True)
        self.d_k = d_model // num_heads  # head dimension = 768 / 12 = 64

        # NOTE: W_Q, W_K, W_V projections live INSIDE nn.MultiheadAttention
        # as self.MHA.in_proj_weight [2304, 768] (Q, K, V stacked).
        # The output projection W_O lives as self.MHA.out_proj.weight [768, 768].
        # We pass x directly — MHA handles Q = xW_Q, K = xW_K, V = xW_V internally.

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0)  # add batch dim: (1, T, d_model)
        seq_len = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len)
        return self.MHA(x, x, x, attn_mask=mask)  # self-attention: Q=K=V=x

Multihead_Attention = MultiheadAttention(d_model=d_model, num_heads=num_heads)
multihead_attention_output = Multihead_Attention(normalized_output)

print(multihead_attention_output[1])

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4582, 0.5418, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2974, 0.3496, 0.3530, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2318, 0.2490, 0.2736, 0.2455, 0.0000, 0.0000, 0.0000],
         [0.2029, 0.1832, 0.1619, 0.1978, 0.2542, 0.0000, 0.0000],
         [0.1533, 0.1353, 0.1557, 0.1521, 0.2119, 0.1917, 0.0000],
         [0.1437, 0.1260, 0.1171, 0.1350, 0.1557, 0.1548, 0.1678]]],
       grad_fn=<MeanBackward1>)


## 4C Layer Norm 2
Second layer norm before the MLP (feed-forward) block. Pre-norm means each sub-block gets a fresh normalized input, making gradient flow much more stable during training.

Pattern per block:
    x = x + Attn(LN₁(x))
    x = x + MLP(LN₂(x))

> Math: Identical formula to Stage 4a with its own learned γ₂, β₂.

In [46]:
# Same Class as Norm4a
class Norm4c(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super(Norm4c, self).__init__()
        self.ln = nn.LayerNorm(
            normalized_shape=d_model,
            eps=eps
        )
    def forward(self, x):
        return self.ln(x)

Norm4C = Norm4c(d_model=d_model)
normalized_output2 = Norm4C(multihead_attention_output[0])


## 4D MLP / Feed-Forward (Linear + Activation)
Two Linear layers with a nonlinear activation between them. This is where the model stores factual associations and knowledge. Expands to $4 \times d_{\text{model}}$ then contracts back. Operates independently on each token position.

### Math

$$W_1 \in \mathbb{R}^{d \times 4d}, \quad W_2 \in \mathbb{R}^{4d \times d}$$

$$\text{MLP}(\mathbf{x}) = W_2 \cdot \text{GELU}(W_1 \mathbf{x} + b_1) + b_2$$

> GELU common in GPT; SiLU/SwiGLU in LLaMA

In [47]:
class LinearActivation(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super(LinearActivation, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, expansion_factor * d_model),
            nn.GELU(),
            nn.Linear(expansion_factor * d_model, d_model),
            nn.Dropout(0.1)
        )
    
    def forward(self, x):
        return self.mlp(x)

Linear_Activation = LinearActivation(d_model=d_model)
linear_activation_output = Linear_Activation(normalized_output2)

## 5 Final Layer Norm
One final normalization after all transformer blocks, before the LM head. Stabilizes the final hidden states before projecting to vocab size.

> Same math as before — normalizes the final (B, T, d_model) tensor per token vector.

In [48]:
# Same Class as Norm4a
class Norm5(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super(Norm5, self).__init__()
        self.ln = nn.LayerNorm(
            normalized_shape=d_model,
            eps=eps
        )
    def forward(self, x):
        return self.ln(x)

norm5 = Norm5(d_model=d_model)
normalized_output_final = norm5(linear_activation_output)


## 6 LM Head (Unembedding)

Projects from $d_{\text{model}} \to |V|$ to get a score (logit) for every possible next token. Often weight-tied to $W_E$ (same matrix shared). Apply softmax to get probabilities, or use argmax for greedy decoding.

### Math

$$W_U \in \mathbb{R}^{d_{\text{model}} \times |V|}$$

$$\text{logits} = \mathbf{x} \cdot W_U$$

> Output: $(B, T, |V|)$ --- $P(\text{next}) = \text{softmax}(\text{logits}[-1])$

In [49]:
class LMHead(nn.Module):
    def __init__(self, d_model, vocab_size):
        super(LMHead, self).__init__()
        self.lm_head = nn.Linear(
            in_features=d_model,
            out_features=vocab_size,
            bias=False
        )

    def forward(self, x):
        return self.lm_head(x)

lm_head = LMHead(d_model=d_model, vocab_size=vocab_size)
lm_output = lm_head(normalized_output_final)

In [50]:
lm_output

tensor([[[ 0.0462,  0.2303,  0.7534,  ..., -0.8939,  0.6157, -0.4041],
         [ 0.1160, -0.2311, -0.5992,  ..., -0.0382,  0.4506, -0.6500],
         [-0.4820, -0.5349,  0.0614,  ..., -0.1607,  0.6847,  0.1532],
         ...,
         [ 0.0412, -0.5935,  0.0594,  ..., -0.4008,  1.1464, -0.4283],
         [ 0.4028, -0.9181, -0.8505,  ..., -0.4398,  0.2954,  0.0645],
         [ 0.4205, -0.2315, -0.2601,  ...,  0.1892,  1.1015,  0.6439]]],
       grad_fn=<UnsafeViewBackward0>)

## 7 Putting It All Together

Each transformer "block" follows this pattern (GPT-2 uses **pre-norm**, meaning LayerNorm comes *before* the sub-layer, not after):

$$\mathbf{h} = \mathbf{x} + \text{Attn}(\text{LN}_1(\mathbf{x}))$$

$$\mathbf{x'} = \mathbf{h} + \text{MLP}(\text{LN}_2(\mathbf{h}))$$

The **residual connections** (the $+ \, \mathbf{x}$ and $+ \, \mathbf{h}$) are critical — they let gradients flow straight through without vanishing, and let each sub-layer learn a *delta* rather than the full representation.

GPT-2 Small stacks **12 identical blocks**, each with its own learned weights. The full forward pass is:

$$\mathbf{x}_0 = W_E[\text{ids}] + W_P[\text{positions}]$$

$$\mathbf{x}_{i+1} = \text{Block}_i(\mathbf{x}_i) \quad \text{for } i = 0, \ldots, 11$$

$$\text{logits} = \text{LN}_f(\mathbf{x}_{12}) \cdot W_U$$

> We reuse every class we built above — `TokenEmbedding`, `PositionalEncoding`, `MultiheadAttention`, `Norm4a`/`Norm4c`, `LinearActivation`, `Norm5`, `LMHead`.

In [51]:
class TransformerBlock(nn.Module):
    """One transformer block: LN1 -> MHA -> residual -> LN2 -> MLP -> residual."""
    def __init__(self, d_model, num_heads):
        super(TransformerBlock, self).__init__()
        self.ln_1 = Norm4a(d_model=d_model)                                  # pre-attention layer norm
        self.attn = MultiheadAttention(d_model=d_model, num_heads=num_heads)  # multi-head self-attention
        self.ln_2 = Norm4c(d_model=d_model)                                  # pre-MLP layer norm
        self.mlp = LinearActivation(d_model=d_model)                          # feed-forward network

    def forward(self, x):
        # x shape: (B, T, d_model)
        # Sub-block 1: attention with residual connection
        attn_output, _ = self.attn(self.ln_1(x))    # (B, T, d_model)
        h = x + attn_output                          # residual connection

        # Sub-block 2: MLP with residual connection
        mlp_output = self.mlp(self.ln_2(h))           # (B, T, d_model)
        out = h + mlp_output                           # residual connection
        return out                                     # (B, T, d_model)


n_layer = 12  # GPT-2 Small has 12 transformer blocks

class GPT2(nn.Module):
    """Full GPT-2 Small: embeddings -> 12 transformer blocks -> LN -> LM head."""
    def __init__(self, vocab_size, d_model, T_max, n_layer, num_heads):
        super(GPT2, self).__init__()
        self.token_embedding = TokenEmbedding(vocab_size=vocab_size, d_model=d_model)
        self.positional_encoding = PositionalEncoding(T_max=T_max, embedding_dim=d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model=d_model, num_heads=num_heads)
            for _ in range(n_layer)
        ])
        self.ln_f = Norm5(d_model=d_model)            # final layer norm
        self.lm_head = LMHead(d_model=d_model, vocab_size=vocab_size)

    def forward(self, input_ids):
        T = input_ids.shape[-1]
        positions = torch.arange(T)                    # (T,)

        # Embeddings: token + position
        x = self.token_embedding(input_ids)            # (B, T, d_model)
        x = x + self.positional_encoding(positions)    # (B, T, d_model)

        # 12 transformer blocks
        for block in self.blocks:
            x = block(x)                               # (B, T, d_model)

        # Final layer norm + LM head
        x = self.ln_f(x)                               # (B, T, d_model)
        logits = self.lm_head(x)                        # (B, T, vocab_size)
        return logits

gpt2 = GPT2(
    vocab_size=vocab_size,
    d_model=d_model,
    T_max=T_max,
    n_layer=n_layer,
    num_heads=num_heads
)

# Verify architecture
print(f"Total parameters: {sum(p.numel() for p in gpt2.parameters()):,}")
print(f"GPT-2 Small should have ~124M parameters")

Total parameters: 163,037,184
GPT-2 Small should have ~124M parameters


## 8 Loading Pretrained Weights

Now the payoff — we copy real GPT-2 weights into our hand-built model. Three things to know:

### Conv1D vs nn.Linear

GPT-2 (HuggingFace) uses `Conv1D`, which stores weight matrices as $[\text{in}, \text{out}]$. PyTorch's `nn.Linear` stores them as $[\text{out}, \text{in}]$. So every weight matrix from `c_attn`, `c_proj`, `c_fc` must be **transposed** (`.T`).

### Packed Q, K, V

GPT-2 packs $W_Q$, $W_K$, $W_V$ into a single matrix `c_attn.weight` of shape $[768, 2304]$. Our `nn.MultiheadAttention` expects `in_proj_weight` of shape $[2304, 768]$. So we transpose: $[768, 2304] \to [2304, 768]$.

### Weight Tying

GPT-2 shares the token embedding matrix $W_E$ with the LM head $W_U$ — "embed" and "unembed" are inverse operations, so they use the same learned matrix.

In [52]:
def load_gpt2_weights(our_model):
    """Load pretrained GPT-2 weights into our hand-built model."""
    # Load the official GPT-2 model (~500MB, cached to ~/.cache/huggingface/)
    hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
    sd = hf_model.state_dict()

    with torch.no_grad():
        # --- Embeddings ---
        our_model.token_embedding.W_E.weight.copy_(
            sd["transformer.wte.weight"]                 # [50257, 768] — same layout, no transpose
        )
        our_model.positional_encoding.W_P.weight.copy_(
            sd["transformer.wpe.weight"]                 # [1024, 768] — same layout, no transpose
        )

        # --- 12 Transformer Blocks ---
        for i in range(12):
            prefix = f"transformer.h.{i}"
            block = our_model.blocks[i]

            # Layer Norm 1 (pre-attention)
            block.ln_1.ln.weight.copy_(sd[f"{prefix}.ln_1.weight"])     # [768]
            block.ln_1.ln.bias.copy_(sd[f"{prefix}.ln_1.bias"])         # [768]

            # Multi-Head Attention
            # c_attn packs Q,K,V: Conv1D [768, 2304] -> transpose -> [2304, 768]
            block.attn.MHA.in_proj_weight.copy_(
                sd[f"{prefix}.attn.c_attn.weight"].T     # [768, 2304].T = [2304, 768]
            )
            block.attn.MHA.in_proj_bias.copy_(
                sd[f"{prefix}.attn.c_attn.bias"]          # [2304] — no transpose for bias
            )
            # c_proj (output projection): Conv1D [768, 768] -> transpose -> [768, 768]
            block.attn.MHA.out_proj.weight.copy_(
                sd[f"{prefix}.attn.c_proj.weight"].T      # [768, 768].T = [768, 768]
            )
            block.attn.MHA.out_proj.bias.copy_(
                sd[f"{prefix}.attn.c_proj.bias"]           # [768]
            )

            # Layer Norm 2 (pre-MLP)
            block.ln_2.ln.weight.copy_(sd[f"{prefix}.ln_2.weight"])     # [768]
            block.ln_2.ln.bias.copy_(sd[f"{prefix}.ln_2.bias"])         # [768]

            # MLP
            # c_fc (expand): Conv1D [768, 3072] -> transpose -> [3072, 768]
            block.mlp.mlp[0].weight.copy_(
                sd[f"{prefix}.mlp.c_fc.weight"].T         # [768, 3072].T = [3072, 768]
            )
            block.mlp.mlp[0].bias.copy_(
                sd[f"{prefix}.mlp.c_fc.bias"]              # [3072]
            )
            # c_proj (contract): Conv1D [3072, 768] -> transpose -> [768, 3072]
            block.mlp.mlp[2].weight.copy_(
                sd[f"{prefix}.mlp.c_proj.weight"].T        # [3072, 768].T = [768, 3072]
            )
            block.mlp.mlp[2].bias.copy_(
                sd[f"{prefix}.mlp.c_proj.bias"]             # [768]
            )

        # --- Final Layer Norm ---
        our_model.ln_f.ln.weight.copy_(sd["transformer.ln_f.weight"])   # [768]
        our_model.ln_f.ln.bias.copy_(sd["transformer.ln_f.bias"])       # [768]

        # --- LM Head (weight-tied to token embedding) ---
        our_model.lm_head.lm_head.weight.copy_(
            sd["transformer.wte.weight"]                 # weight tying! same as W_E
        )

    print("All weights loaded successfully!")
    return our_model

gpt2 = load_gpt2_weights(gpt2)
gpt2.eval()  # disable dropout for deterministic inference

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All weights loaded successfully!


GPT2(
  (token_embedding): TokenEmbedding(
    (W_E): Embedding(50257, 768)
  )
  (positional_encoding): PositionalEncoding(
    (W_P): Embedding(1024, 768)
  )
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln_1): Norm4a(
        (ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (attn): MultiheadAttention(
        (MHA): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
      )
      (ln_2): Norm4c(
        (ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (mlp): LinearActivation(
        (mlp): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (ln_f): Norm5(
    (ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (

## 9 Decoding (Logits → Predicted Token)

The model outputs **logits**: a raw score for every token in the vocabulary at every position. To get the predicted next token:

$$P(\text{next token}) = \text{softmax}(\text{logits}[-1])$$

$$\hat{y} = \arg\max_i \, P_i$$

We only care about the **last position's** logits — that's the model's prediction for what comes *after* the final input token. This is **greedy decoding** (always pick the highest-probability token).

In [53]:
# Run our hand-built GPT-2 on the same input from Section 1
with torch.no_grad():
    logits = gpt2(input_ids.unsqueeze(0))               # (1, T, vocab_size) — add batch dim

# Grab logits for the LAST token position
last_logits = logits[0, -1, :]                           # (vocab_size,) = (50257,)

# Greedy decode: argmax
predicted_token_id = torch.argmax(last_logits).item()
predicted_token = Tokenizer.decode(predicted_token_id)

print(f'Input:  "{text}"')
print(f'Predicted next token: "{predicted_token}" (id={predicted_token_id})')
print()

# Top-5 predictions with probabilities
probs = torch.softmax(last_logits, dim=-1)
top5 = torch.topk(probs, 5)
print("Top 5 predictions:")
for i, (prob, idx) in enumerate(zip(top5.values, top5.indices)):
    token = Tokenizer.decode(idx.item())
    print(f'  {i+1}. "{token}" (id={idx.item()}, prob={prob.item():.4f})')

Input:  "Hello there, who are you?"
Predicted next token: "
" (id=198)

Top 5 predictions:
  1. "
" (id=198, prob=0.2001)
  2. " I" (id=314, prob=0.0815)
  3. " What" (id=1867, prob=0.0296)
  4. " You" (id=921, prob=0.0248)
  5. " Are" (id=4231, prob=0.0177)


## What I Learned

- **Residual connections** are what make deep transformers trainable — each block learns a small *delta*, not the full representation
- GPT-2 uses **pre-norm** (LayerNorm before attention/MLP), not post-norm
- `nn.MultiheadAttention` packs $W_Q$, $W_K$, $W_V$ into a single `in_proj_weight` matrix of shape $[3d, d]$
- GPT-2's Conv1D stores weights **transposed** compared to `nn.Linear` — every weight must be `.T`'d when loading
- **Weight tying**: the LM head reuses the token embedding matrix, which makes intuitive sense — "embed" and "unembed" are inverse operations
- With random weights the output is random; with real weights, the exact same architecture produces coherent English predictions